In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "ndnsim-research-main" / "ndn-research" / "results"
SAVE_DIR = PROJECT_ROOT / "processed"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TOPOLOGIES = ["dfn", "dumbbell", "tree"]
SCENARIOS  = ["normal", "ifa", "cp"]

def data_file(topo, scenario, ftype):
    ext = "csv" if ftype == "ground-truth" else "txt"
    return DATA_DIR / f"{topo}-{scenario}-{ftype}.{ext}"

print("Paths ready.")
print(f"DATA_DIR : {DATA_DIR}")
print(f"SAVE_DIR : {SAVE_DIR}")

Paths ready.
DATA_DIR : /Users/ankitpokhrel/Desktop/minor_project_refactored/NDNsim/ndnsim-research-main/ndn-research/results
SAVE_DIR : /Users/ankitpokhrel/Desktop/minor_project_refactored/NDNsim/processed


In [2]:
print(f"{'File':<45} {'Exists':>6} {'Size (KB)':>10}")
print("-" * 65)
missing = []
for topo in TOPOLOGIES:
    for scenario in SCENARIOS:
        for ftype in ["rate-trace", "cs-trace", "app-delays", "ground-truth"]:
            p = data_file(topo, scenario, ftype)
            exists = p.exists()
            size = f"{p.stat().st_size / 1024:.1f}" if exists else "—"
            flag = "✓" if exists else "✗ MISSING"
            print(f"{p.name:<45} {flag:>6} {size:>10}")
            if not exists:
                missing.append(p.name)

print(f"\n{len(missing)} file(s) missing: {missing if missing else 'none'}")

File                                          Exists  Size (KB)
-----------------------------------------------------------------
dfn-normal-rate-trace.txt                          ✓    19515.2
dfn-normal-cs-trace.txt                            ✓      269.9
dfn-normal-app-delays.txt                          ✓     2172.2
dfn-normal-ground-truth.csv                        ✓        6.3
dfn-ifa-rate-trace.txt                             ✓    17724.9
dfn-ifa-cs-trace.txt                               ✓      272.2
dfn-ifa-app-delays.txt                             ✓     2172.2
dfn-ifa-ground-truth.csv                           ✓        5.5
dfn-cp-rate-trace.txt                              ✓    29955.9
dfn-cp-cs-trace.txt                                ✓      272.8
dfn-cp-app-delays.txt                              ✓     7894.9
dfn-cp-ground-truth.csv                            ✓        5.2
dumbbell-normal-rate-trace.txt                     ✓    14731.7
dumbbell-normal-cs-trace.txt          

In [3]:
def load_rate_trace(topo, scenario):
    p = data_file(topo, scenario, "rate-trace")
    rt = pd.read_csv(p, sep="\t", comment="#")
    rt.columns = rt.columns.str.strip()

    KEEP_TYPES = {
        "InInterests", "OutInterests", "InData",
        "InNacks", "InSatisfiedInterests", "InTimedOutInterests"
    }
    rt = rt[rt["Type"].isin(KEEP_TYPES)][["Time", "Node", "Type", "Packets"]]

    rt_pivot = (
        rt.groupby(["Time", "Node", "Type"])["Packets"]
        .sum()
        .unstack("Type")
        .fillna(0)
        .reset_index()
    )
    rt_pivot.columns.name = None
    return rt_pivot

# Test
sample = load_rate_trace("dfn", "normal")
print("Shape:", sample.shape)
print("Columns:", sample.columns.tolist())
print("Time range:", sample["Time"].min(), "→", sample["Time"].max())
sample.head(3)

Shape: (7188, 8)
Columns: ['Time', 'Node', 'InData', 'InInterests', 'InNacks', 'InSatisfiedInterests', 'InTimedOutInterests', 'OutInterests']
Time range: 1 → 599


,Time,Node,InData,InInterests,InNacks,InSatisfiedInterests,InTimedOutInterests,OutInterests
0,1,0,28.0,28.0,0.0,29.6,0.0,28.0
1,1,1,0.0,0.0,0.0,1.6,0.0,0.0
2,1,2,28.0,36.8,0.0,34.4,0.0,28.0


In [4]:
def load_cs_trace(topo, scenario):
    p = data_file(topo, scenario, "cs-trace")
    cs = pd.read_csv(p, sep="\t", comment="#")
    cs.columns = cs.columns.str.strip()
    cs = cs[["Time", "Node", "Type", "Packets"]]

    cs_pivot = (
        cs.groupby(["Time", "Node", "Type"])["Packets"]
        .sum()
        .unstack("Type")
        .fillna(0)
        .reset_index()
    )
    cs_pivot.columns.name = None
    return cs_pivot

sample_cs = load_cs_trace("dfn", "normal")
print("Shape:", sample_cs.shape)
sample_cs.head(3)

Shape: (7188, 4)


,Time,Node,CacheHits,CacheMisses
0,1,0,0,36
1,1,1,0,1
2,1,2,9,42


In [5]:
def load_app_delays(topo, scenario):
    p = data_file(topo, scenario, "app-delays")
    ad = pd.read_csv(p, sep="\t", comment="#")
    ad.columns = ad.columns.str.strip()
    ad = ad[ad["Type"] == "FullDelay"].copy()

    ad["Time"] = ad["Time"].apply(np.floor).astype(int).clip(lower=1)

    agg = (
        ad.groupby(["Time", "Node"])
        .agg(
            delay_mean  = ("DelayS",    "mean"),
            delay_p95   = ("DelayS",    lambda x: np.percentile(x, 95)),
            retx_mean   = ("RetxCount", "mean"),
            hop_mean    = ("HopCount",  "mean"),
            n_satisfied = ("DelayS",    "count")
        )
        .reset_index()
    )
    return agg

sample_ad = load_app_delays("dfn", "normal")
print("Shape:", sample_ad.shape)
print("Nodes present (consumers):", sorted(sample_ad["Node"].unique()))
sample_ad.head(3)

Shape: (3594, 7)
Nodes present (consumers): [np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]


,Time,Node,delay_mean,delay_p95,retx_mean,hop_mean,n_satisfied
0,1,6,0.049057,0.062626,1.0,3.0,20
1,1,7,0.040264,0.064333,1.0,3.0,20
2,1,8,0.043050,0.063479,1.0,3.0,20


In [6]:
def load_ground_truth(topo, scenario):
    gt = pd.read_csv(data_file(topo, scenario, "ground-truth"))
    gt.columns = gt.columns.str.strip().str.lower()
    gt = gt.rename(columns={"time": "Time"})

    # Ground truth is 0-indexed (t=0 to t=599).
    # Rate/CS traces are 1-indexed (t=1 to t=599).
    # Shift so labels align with trace timestamps.
    # t=0 in GT → t=1 in traces, t=299 → t=300 (last normal), t=300 → t=301 (first attack)
    gt["Time"] = gt["Time"] + 1

    # Keep only rows that exist in trace data (t=1 to t=599)
    gt = gt[(gt["Time"] >= 1) & (gt["Time"] <= 599)]

    return gt

# Sanity check
gt_test = load_ground_truth("dfn", "ifa")
print(gt_test.shape)
print(gt_test["label"].value_counts())
print("\nBoundary check (should be normal at 300, ifa at 301):")
print(gt_test[gt_test["Time"].isin([299, 300, 301, 302])])

(599, 2)
label
normal    300
ifa       299
Name: count, dtype: int64

Boundary check (should be normal at 300, ifa at 301):
     Time   label
298   299  normal
299   300  normal
300   301     ifa
301   302     ifa


In [7]:
CONSUMER_NODES = {}
for topo in TOPOLOGIES:
    ad = pd.read_csv(data_file(topo, "normal", "app-delays"), sep="\t", comment="#")
    ad.columns = ad.columns.str.strip()
    CONSUMER_NODES[topo] = set(ad["Node"].unique())

for topo, nodes in CONSUMER_NODES.items():
    rt = pd.read_csv(data_file(topo, "normal", "rate-trace"), sep="\t", comment="#")
    rt.columns = rt.columns.str.strip()
    all_nodes = set(rt["Node"].unique())
    router_nodes = all_nodes - nodes
    print(f"{topo}: consumers={sorted(nodes)}, routers={sorted(router_nodes)}")

dfn: consumers=[np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)], routers=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


dumbbell: consumers=[np.int64(0), np.int64(1), np.int64(2)], routers=[np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
tree: consumers=[np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)], routers=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [8]:
def engineer_features(topo, scenario):
    rt = load_rate_trace(topo, scenario)
    cs = load_cs_trace(topo, scenario)
    ad = load_app_delays(topo, scenario)
    gt = load_ground_truth(topo, scenario)

    df = (rt
          .merge(cs, on=["Time", "Node"], how="left")
          .merge(ad, on=["Time", "Node"], how="left"))

    # Fill router NaNs (no app-delay data by design)
    delay_cols = ["delay_mean", "delay_p95", "retx_mean", "hop_mean", "n_satisfied"]
    df[delay_cols] = df[delay_cols].fillna(0)

    # Features
    total_cs = df["CacheHits"] + df["CacheMisses"]
    df["cache_hit_ratio"] = np.where(total_cs > 0, df["CacheHits"] / total_cs, 0.0)

    df["satisfaction_ratio"] = np.where(
        df["InInterests"] > 0,
        df["InSatisfiedInterests"] / df["InInterests"], 0.0).clip(0, 1)

    df["timeout_ratio"] = np.where(
        df["InInterests"] > 0,
        df["InTimedOutInterests"] / df["InInterests"], 0.0).clip(0, 1)

    df["nack_ratio"] = np.where(
        df["InInterests"] > 0,
        df["InNacks"] / df["InInterests"], 0.0).clip(0, 1)

    df["interest_amp"] = np.where(
        df["InInterests"] > 0,
        df["OutInterests"] / df["InInterests"], 1.0)

    df["data_ratio"] = np.where(
        df["OutInterests"] > 0,
        df["InData"] / df["OutInterests"], 0.0).clip(0, 1)

    # Node role
    df["role"] = df["Node"].apply(
        lambda n: "consumer" if n in CONSUMER_NODES[topo] else "router")

    # Labels
    df = df.merge(gt[["Time", "label"]], on="Time", how="left")
    df["label"] = df["label"].fillna("normal")

    df["topology"] = topo
    df["scenario"] = scenario
    return df

# Test
test_df = engineer_features("dfn", "normal")
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())
print("NaN remaining:", test_df.isna().sum()[test_df.isna().sum() > 0].to_dict())
test_df.head(3)

Shape: (7188, 25)
Columns: ['Time', 'Node', 'InData', 'InInterests', 'InNacks', 'InSatisfiedInterests', 'InTimedOutInterests', 'OutInterests', 'CacheHits', 'CacheMisses', 'delay_mean', 'delay_p95', 'retx_mean', 'hop_mean', 'n_satisfied', 'cache_hit_ratio', 'satisfaction_ratio', 'timeout_ratio', 'nack_ratio', 'interest_amp', 'data_ratio', 'role', 'label', 'topology', 'scenario']
NaN remaining: {}


,Time,Node,InData,InInterests,InNacks,InSatisfiedInterests,InTimedOutInterests,OutInterests,CacheHits,CacheMisses,...,cache_hit_ratio,satisfaction_ratio,timeout_ratio,nack_ratio,interest_amp,data_ratio,role,label,topology,scenario
0,1,0,28.0,28.0,0.0,29.6,0.0,28.0,0,36,...,0.000000,1.000000,0.0,0.0,1.00000,1.0,router,normal,dfn,normal
1,1,1,0.0,0.0,0.0,1.6,0.0,0.0,0,1,...,0.000000,0.000000,0.0,0.0,1.00000,0.0,router,normal,dfn,normal
2,1,2,28.0,36.8,0.0,34.4,0.0,28.0,9,42,...,0.176471,0.934783,0.0,0.0,0.76087,1.0,router,normal,dfn,normal


In [9]:
all_dfs = []
for topo in TOPOLOGIES:
    for scenario in SCENARIOS:
        try:
            df = engineer_features(topo, scenario)
            all_dfs.append(df)
            print(f"{topo}-{scenario}: {df.shape} | labels: {df['label'].value_counts().to_dict()}")
        except Exception as e:
            print(f"ERROR {topo}-{scenario}: {e}")

full = pd.concat(all_dfs, ignore_index=True)
print(f"\nFull dataset: {full.shape}")
print(full["label"].value_counts())

dfn-normal: (7188, 25) | labels: {'normal': 7188}


dfn-ifa: (7188, 25) | labels: {'normal': 3600, 'ifa': 3588}


dfn-cp: (7128, 25) | labels: {'cp': 3588, 'normal': 3540}
dumbbell-normal: (5990, 25) | labels: {'normal': 5990}


dumbbell-ifa: (5990, 25) | labels: {'normal': 3000, 'ifa': 2990}


dumbbell-cp: (5870, 25) | labels: {'cp': 2990, 'normal': 2880}


tree-normal: (7188, 25) | labels: {'normal': 7188}


tree-ifa: (7188, 25) | labels: {'normal': 3600, 'ifa': 3588}


tree-cp: (7188, 25) | labels: {'normal': 3600, 'cp': 3588}

Full dataset: (60918, 25)
label
normal    40586
ifa       10166
cp        10166
Name: count, dtype: int64


In [10]:
# Full dataset
full.to_csv(SAVE_DIR / "full_dataset.csv", index=False)
print(f"Saved full_dataset.csv — {full.shape}")

# Normal only (for baseline in detection)
train_normal = full[full["scenario"] == "normal"]
train_normal.to_csv(SAVE_DIR / "train_normal.csv", index=False)
print(f"Saved train_normal.csv — {train_normal.shape}")

# Per topology-scenario splits
for topo in TOPOLOGIES:
    for scenario in SCENARIOS:
        subset = full[(full["topology"] == topo) & (full["scenario"] == scenario)]
        fname = SAVE_DIR / f"{topo}_{scenario}.csv"
        subset.to_csv(fname, index=False)
        print(f"Saved {fname.name} — {subset.shape}")

print(f"\n✓ All files saved to {SAVE_DIR}")

Saved full_dataset.csv — (60918, 25)
Saved train_normal.csv — (20366, 25)


Saved dfn_normal.csv — (7188, 25)
Saved dfn_ifa.csv — (7188, 25)
Saved dfn_cp.csv — (7128, 25)
Saved dumbbell_normal.csv — (5990, 25)


Saved dumbbell_ifa.csv — (5990, 25)
Saved dumbbell_cp.csv — (5870, 25)
Saved tree_normal.csv — (7188, 25)


Saved tree_ifa.csv — (7188, 25)
Saved tree_cp.csv — (7188, 25)

✓ All files saved to /Users/ankitpokhrel/Desktop/minor_project_refactored/NDNsim/processed


In [11]:
print(f"{'File':<35} {'Rows':>8} {'Cols':>6}")
print("-" * 52)
for f in sorted(SAVE_DIR.glob("*.csv")):
    df_check = pd.read_csv(f, nrows=1)
    nrows = sum(1 for _ in open(f)) - 1
    print(f"{f.name:<35} {nrows:>8} {len(df_check.columns):>6}")

File                                    Rows   Cols
----------------------------------------------------
detection_latency.csv                     68      6
dfn_cp.csv                              7128     25
dfn_ifa.csv                             7188     25
dfn_normal.csv                          7188     25
dumbbell_cp.csv                         5870     25
dumbbell_ifa.csv                        5990     25
dumbbell_normal.csv                     5990     25
full_dataset.csv                       60918     25
full_scored_global.csv                 60918     31
full_scored_pertopo.csv                60918     31
ks_results.csv                           748      7
train_normal.csv                       20366     25
tree_cp.csv                             7188     25
tree_ifa.csv                            7188     25
tree_normal.csv                         7188     25


In [12]:
# import pandas as pd
# from pathlib import Path

# DATA_DIR = PROJECT_ROOT / "ndnsim-research-main" / "ndn-research" / "results"

# for topo in ["dfn", "dumbbell", "tree"]:
#     for scenario in ["ifa", "cp"]:
#         gt = pd.read_csv(DATA_DIR / f"{topo}-{scenario}-ground-truth.csv")
#         gt.columns = gt.columns.str.strip().str.lower()
#         counts = gt["label"].value_counts().to_dict()
#         attack_start = gt[gt["label"] != "normal"]["time"].min()
#         print(f"{topo}-{scenario}: {counts} | attack starts at t={attack_start}")